# GPU Stress Test

Benchmarks matrix-multiplication throughput (GFLOPS) for three ML frameworks:
**PyTorch**, **TensorFlow**, and **CuPy**.

## Setup
Install dependencies before opening this notebook:
```bash
uv pip install -e .
```

## GPU availability note
The saved cell outputs below were captured on a machine **without a CUDA-capable GPU**
(results reflect CPU performance). On a machine with an NVIDIA GPU, PyTorch and TensorFlow
should be reinstalled with CUDA support and CuPy should be installed (`cupy-cuda12x`).
Apple Silicon users will see MPS detected but CuPy is not supported on MPS.

In [ ]:
# =============================================================================
# INSTALLATION DIAGNOSTICS
# =============================================================================

def check_gpu_setup():
    """Full diagnostic of available GPUs and ML/DL libraries."""
    print("GPU/ML INSTALLATION DIAGNOSTIC")
    print("=" * 60)

    # 1. CUDA via nvidia-smi
    print("\nCUDA CHECK")
    print("-" * 30)
    try:
        import subprocess
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        if result.returncode == 0:
            lines = result.stdout.split('\n')
            gpu_lines = [l for l in lines if any(k in l for k in ('GeForce', 'RTX', 'GTX'))]
            print(f"NVIDIA Driver: installed")
            print(f"GPUs detected by nvidia-smi: {len(gpu_lines)}")
            for i, line in enumerate(gpu_lines):
                name = line.split('|')[1].strip() if '|' in line else 'unknown GPU'
                print(f"   GPU {i}: {name}")
        else:
            print("NVIDIA Driver: not available")
    except FileNotFoundError:
        print("nvidia-smi: not found")
    except Exception as e:
        print(f"Error checking CUDA: {e}")

    # 2. PyTorch
    print("\nPYTORCH")
    print("-" * 30)
    try:
        import torch
        print(f"PyTorch: v{torch.__version__}")
        if torch.cuda.is_available():
            n = torch.cuda.device_count()
            print(f"CUDA available: yes ({n} GPU(s))")
            for i in range(n):
                name = torch.cuda.get_device_name(i)
                mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
                print(f"   GPU {i}: {name} ({mem:.1f} GB)")
        else:
            print("CUDA available: no")
        if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            print("MPS (Apple Silicon): available")
        print(f"CUDA build version: {torch.version.cuda}")
    except ImportError:
        print("PyTorch: not installed")
    except Exception as e:
        print(f"PyTorch error: {e}")

    # 3. TensorFlow
    print("\nTENSORFLOW")
    print("-" * 30)
    try:
        import tensorflow as tf
        print(f"TensorFlow: v{tf.__version__}")
        print(f"Built with CUDA: {tf.test.is_built_with_cuda()}")
        devices = tf.config.list_physical_devices()
        print(f"Physical devices: {len(devices)}")
        for d in devices:
            print(f"   {d}")
        gpus = tf.config.list_physical_devices('GPU')
        print(f"GPUs detected: {len(gpus)}")
        print(f"CUDA runtime: {tf.sysconfig.get_build_info().get('cuda_version', 'n/a')}")
        print(f"cuDNN: {tf.sysconfig.get_build_info().get('cudnn_version', 'n/a')}")
    except ImportError:
        print("TensorFlow: not installed")
    except Exception as e:
        print(f"TensorFlow error: {e}")

    # 4. CuPy
    print("\nCUPY")
    print("-" * 30)
    try:
        import cupy as cp
        print(f"CuPy: v{cp.__version__}")
        n = cp.cuda.runtime.getDeviceCount()
        print(f"CuPy devices: {n}")
        for i in range(n):
            with cp.cuda.Device(i):
                props = cp.cuda.runtime.getDeviceProperties(i)
                name = props['name'].decode('ascii')
                mem = props['totalGlobalMem'] / 1024**3
                print(f"   GPU {i}: {name} ({mem:.1f} GB)")
    except ImportError:
        print("CuPy: not installed  (install cupy-cuda12x for CUDA 12.x)")
    except Exception as e:
        print(f"CuPy error: {e}")

check_gpu_setup()

In [ ]:
# =============================================================================
# PYTORCH BENCHMARK
# =============================================================================

import time
import numpy as np

def test_pytorch_gpu():
    """Matrix multiplication benchmark for PyTorch (CUDA or CPU fallback)."""
    print("PYTORCH - matrix multiplication benchmark")
    print("-" * 40)
    try:
        import torch

        if torch.cuda.is_available():
            device = torch.device('cuda')
            print(f"GPU: {torch.cuda.get_device_name(0)}")
            print(f"Total GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
        else:
            device = torch.device('cpu')
            print("No CUDA GPU detected — running on CPU")

        size = 20000
        print(f"\nCreating {size}x{size} matrices on {device}...")
        a = torch.randn(size, size, device=device)
        b = torch.randn(size, size, device=device)

        print("Warming up...")
        for _ in range(3):
            _ = torch.mm(a, b)
        if device.type == 'cuda':
            torch.cuda.synchronize()

        print("Running 5 multiplications...")
        start = time.time()
        for _ in range(5):
            result = torch.mm(a, b)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        elapsed = time.time() - start

        gflops = (5 * 2 * size**3) / (elapsed * 1e9)
        print(f"Total time:   {elapsed:.3f} s")
        print(f"Per multiply: {elapsed/5:.3f} s")
        print(f"Throughput:   {gflops:.2f} GFLOPS")

        if device.type == 'cuda':
            used = torch.cuda.memory_allocated() / 1024**3
            print(f"GPU memory used: {used:.2f} GB")

        print("PyTorch benchmark complete\n")
    except ImportError:
        print("PyTorch not installed")
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
test_pytorch_gpu()

In [ ]:
# =============================================================================
# TENSORFLOW BENCHMARK
# =============================================================================

def test_tensorflow_gpu():
    """Matrix multiplication benchmark for TensorFlow (CUDA or CPU fallback)."""
    print("TENSORFLOW - matrix multiplication benchmark")
    print("-" * 40)
    try:
        import tensorflow as tf

        gpus = tf.config.list_physical_devices('GPU')
        print(f"GPUs available: {len(gpus)}")
        if gpus:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            target = '/GPU:0'
        else:
            print("No GPU detected — running on CPU")
            target = '/CPU:0'

        size = 2000
        print(f"\nCreating {size}x{size} tensors...")
        with tf.device(target):
            a = tf.random.normal([size, size])
            b = tf.random.normal([size, size])
        print(f"Tensors on: {a.device}")

        print("Warming up...")
        for _ in range(3):
            _ = tf.linalg.matmul(a, b)

        print("Running 5 multiplications...")
        start = time.time()
        with tf.device(target):
            for _ in range(5):
                result = tf.linalg.matmul(a, b)
        elapsed = time.time() - start

        gflops = (5 * 2 * size**3) / (elapsed * 1e9)
        print(f"Total time:   {elapsed:.3f} s")
        print(f"Per multiply: {elapsed/5:.3f} s")
        print(f"Throughput:   {gflops:.2f} GFLOPS")
        print("TensorFlow benchmark complete\n")
    except ImportError:
        print("TensorFlow not installed")
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
test_tensorflow_gpu()

In [ ]:
# =============================================================================
# CUPY BENCHMARK  (requires CUDA; skipped gracefully on CPU-only machines)
# =============================================================================

def test_cupy_gpu():
    """Matrix multiplication benchmark for CuPy, with CPU comparison."""
    print("CUPY - GPU array benchmark")
    print("-" * 40)
    try:
        import cupy as cp

        print(f"CuPy: v{cp.__version__}")
        print(f"CUDA runtime: {cp.cuda.runtime.runtimeGetVersion()}")

        size = 2000
        print(f"\nCreating {size}x{size} GPU arrays...")
        a = cp.random.randn(size, size)
        b = cp.random.randn(size, size)

        print("Warming up...")
        for _ in range(3):
            _ = cp.dot(a, b)
        cp.cuda.Stream.null.synchronize()

        print("Running 5 multiplications (GPU)...")
        start = time.time()
        for _ in range(5):
            result = cp.dot(a, b)
        cp.cuda.Stream.null.synchronize()
        gpu_time = time.time() - start

        gflops = (5 * 2 * size**3) / (gpu_time * 1e9)
        print(f"GPU total time:   {gpu_time:.3f} s")
        print(f"GPU per multiply: {gpu_time/5:.3f} s")
        print(f"Throughput:       {gflops:.2f} GFLOPS")

        # CPU comparison
        print("\nRunning equivalent multiply on CPU for comparison...")
        a_cpu = cp.asnumpy(a)
        b_cpu = cp.asnumpy(b)
        start = time.time()
        _ = np.dot(a_cpu, b_cpu)
        cpu_time = time.time() - start
        print(f"CPU time: {cpu_time:.3f} s")
        print(f"GPU speedup: {cpu_time/gpu_time:.1f}x")

        print("CuPy benchmark complete\n")
    except ImportError:
        print("CuPy not installed  (install cupy-cuda12x for CUDA 12.x)")
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
test_cupy_gpu()

In [ ]:
# =============================================================================
# RUN ALL BENCHMARKS
# =============================================================================

def run_all_gpu_tests():
    """Run all three framework benchmarks sequentially."""
    print("STARTING ALL GPU BENCHMARKS")
    print("=" * 50)
    check_gpu_setup()
    test_pytorch_gpu()
    test_tensorflow_gpu()
    test_cupy_gpu()
    print("ALL BENCHMARKS COMPLETE")
    print("=" * 50)